In [3]:
# 환경 설정 및 라이브러리 설치
!pip install -q openai langchain langchain-openai langchain-community faiss-cpu \
    rank_bm25 pandas numpy matplotlib gradio python-dotenv tiktoken

In [ ]:
!pip install -U ddgs # 덕덕고

In [5]:
import os, json, math
from datetime import datetime, timedelta
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
# from dotenv import load_dotenv
# load_dotenv()
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
MODEL = "gpt-4o-mini"

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool, StructuredTool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

llm = ChatOpenAI(model="gpt-4o-mini")

In [7]:
# @tool
# search wikipedia duckduckgo

In [ ]:
# 4 5 -> M * N = 총 20개
  # 기존 방식: 4개의 AI 서비스와 5개의 도구(a,b,c,d,e)를 각각 연결하면 20번의 개발이 필요함.

# 1번 서비스      a
# 2번 서비스  mcp b
# 3번 서비스      c d e
# 4번 서비스(생략됨)
  # (이 부분은 AI 서비스들(1,2,3,4)과 도구들(a,b,c,d,e) 사이에 mcp가 허브 역할을 한다는 구조도)

# a, b, c, d, e 툴들은 mcp랑만 연결
  # 도구들은 개별 AI API를 신경 쓸 필요 없이 MCP 서버 표준에 맞춰서 한 번만 개발하면 됨.

# M + N 은 9개만 개발하면 됨
  # AI 서비스 4개가 MCP를 지원하도록 만들고(4), 도구 5개가 MCP를 지원하도록 만들면(5),
  # 총 9개의 작업만으로 모든 서비스와 도구가 자유롭게 통신할 수 있음

In [ ]:
# tavily-python : 검색용 API

In [ ]:
!pip install tavily-python wikipedia duckduckgo-search numexpr

In [9]:
# tool은 그냥 함수이긴 한데, 랭체인을 사용하기 전, 쿼리를 사용해 답변을 주기 전에
# 툴을 호출하여 결과를 이용함
@tool
def example_tool(query):
  """example tool"""
  return f"결과: {query}"

In [10]:
example_tool

StructuredTool(name='example_tool', description='example tool', args_schema=<class 'langchain_core.utils.pydantic.example_tool'>, func=<function example_tool at 0x7812abbfd8a0>)

In [11]:
example_tool.invoke({'query' : 'test'})

'결과: test'

In [12]:
from datetime import datetime

In [13]:
@tool
def get_current_time():
  """return current datetime"""
  return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

@tool
def string_length(text):
  """return string length"""
  return len(text)


In [14]:
for t in [get_current_time, string_length]:
  print(f"tools : {t.name}: {t.description}")
  schema = t.args_schema.schema()
  print(f" args: {schema.get('properties', {})}")

tools : get_current_time: return current datetime
 args: {}
tools : string_length: return string length
 args: {'text': {'title': 'Text'}}


/tmp/ipykernel_20162/1717657182.py:3: PydanticDeprecatedSince20: The `schema` method is deprecated; use `model_json_schema` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  schema = t.args_schema.schema()


In [15]:
get_current_time.invoke({})

'2026-04-22 11:13:18'

In [16]:
string_length.invoke({'text': 'hello'})

5

In [17]:
from langchain_community.tools.tavily_search import TavilySearchResults

In [19]:
# Tavily는 api-key 다운 필요# Tavily는 api-key 다운 필요
  # 여기서 가져올것 https://app.tavily.com/
os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')

In [20]:
search_tool = TavilySearchResults(max_results = 3)

/tmp/ipykernel_20162/1420787621.py:1: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(max_results = 3)


In [21]:
search_tool.name

'tavily_search_results_json'

In [22]:
search_tool.description

'A search engine optimized for comprehensive, accurate, and trusted results. Useful for when you need to answer questions about current events. Input should be a search query.'

In [25]:
results = search_tool.invoke({'query': '2026년 한국 ETF 시장 동향'}) # max_results = 3 으로 했으니까 총 3개 반환
results

[{'title': '2026 ETF 투자 전략 총정리 — 초보자부터 고수까지, 유망 테마·절세',
  'url': 'https://baehoon.tistory.com/137',
  'content': "## 2026년 ETF 시장 현황 — 370조 돌파, 400조 시대 눈앞\n\n국내 ETF 시장이 무섭게 성장하고 있습니다. 2026년 2월 기준 순자산총액 370조 원을 돌파했으며, 2026년 1분기 내 400조 원 달성이 유력한 상황입니다.\n\n 국내 ETF 순자산총액 성장 추이 차트 \n\n국내 ETF 시장은 2020년 52조에서 2026년 2월 370조로 급성장했습니다\n\n \n\n불과 6년 전인 2020년에는 52조 원이었던 시장이 7배 이상 성장한 셈입니다. 2025년 1월에 300조 원을 처음 돌파한 지 1년여 만에 70조 원이 추가로 유입되었습니다.\n\n### 왜 ETF 시장이 이렇게 빠르게 커지고 있을까?\n\n| 성장 요인 | 설명 |\n --- |\n| AI·테마 ETF 열풍 | AI, 반도체, 로봇 등 테마형 ETF에 자금이 집중 |\n| 월배당 ETF 인기 | 매달 현금흐름을 받을 수 있는 월배당 ETF 수요 급증 |\n| 저비용 구조 | 펀드 대비 보수가 10분의 1 수준으로 저렴 |\n| ISA 등 절세 계좌 | ISA, IRP 등 절세 계좌에서 ETF 투자가 활발 |\n\n> 국내 ETF 성장률은 글로벌 평균(연 20%)의 2배 이상! 2026년 1분기 내 400조 원 달성이 유력합니다.\n\n한마디로, 지금은 ETF 투자를 시작하기에 가장 좋은 시기입니다. 시장이 커질수록 상품의 종류와 유동성이 풍부해져 투자자에게 유리한 환경이 만들어지기 때문입니다.\n\n## 8대 운용사가 꼽은 2026년 ETF 4대 키워드 [...] ## ETF란? 주식과 펀드의 장점을 합친 투자 상품\n\nETF는 Exchange Traded Fund의 약자로, 한국어로는 '상장지수펀드'라고 합니다. 쉽게 말해 여러 종목을 한 바구니에 담아 주식처럼 

In [28]:
for i, r in enumerate(results, 1):
  print(f"[{i}] {r.get('url', 'N/A')}")
  print(f"   {r.get('content', '')[:100]}...")
  print("=="*20)
# 이러한 결과값을 이용해 LLM에 피드하여 더 좋은 결과값을 반환할 수 있을 것

[1] https://baehoon.tistory.com/137
   ## 2026년 ETF 시장 현황 — 370조 돌파, 400조 시대 눈앞

국내 ETF 시장이 무섭게 성장하고 있습니다. 2026년 2월 기준 순자산총액 370조 원을 돌파했으며,...
[2] https://ddorm.tistory.com/entry/top-etf-investing-2026-1028
   ## 국장 랠리의 중심: 한국 ETF 상위 5개

최근 코스피가 6,000선을 넘나드는 역사적인 랠리를 펼치면서 국내 ETF 시장도 엄청난 활기를 띠고 있어요.  
특히 삼성전자와...
[3] https://magazine.hankyung.com/money/article/202601187642c
   자산운용 시장은 상장지수펀드(ETF)를 중심으로 재편 중이다. 전통적인 뮤추얼 펀드에서 이탈한 자금이 ETF로 이동하면서 연간 자금 순유입 규모는 사상 최대 규모로 성장했다. 글로...


In [29]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, ToolMessage

In [30]:
llm_with_search = llm.bind_tools([search_tool])

In [31]:
query = "2026년 나스닥 전망은?"
messages = [HumanMessage(content = query)]
response = llm_with_search.invoke(messages)

In [32]:
response
#  tool_calls=[{'name': 'tavily_search_results_json',
# 'args': {'query': '2026년 나스닥 전망'}, 'id': 'call_gPU64EjBOd9IKTLgeJbaTLcl', 'type': 'tool_call'}]

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 88, 'total_tokens': 112, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_38babba494', 'id': 'chatcmpl-DXQ0bxQVsOrUZfpqGfkqCNfs2fAqJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019db4f0-c78b-7ce0-a7f7-c3fbe4786186-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': '2026년 나스닥 전망'}, 'id': 'call_gPU64EjBOd9IKTLgeJbaTLcl', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 88, 'output_tokens': 24, 'total_tokens': 112, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoni

In [33]:
response.tool_calls # 이용하여 최종 답변 불러오는 코드

[{'name': 'tavily_search_results_json',
  'args': {'query': '2026년 나스닥 전망'},
  'id': 'call_gPU64EjBOd9IKTLgeJbaTLcl',
  'type': 'tool_call'}]

In [35]:
response.tool_calls[0]

{'name': 'tavily_search_results_json',
 'args': {'query': '2026년 나스닥 전망'},
 'id': 'call_gPU64EjBOd9IKTLgeJbaTLcl',
 'type': 'tool_call'}

In [ ]:
if response.tool_calls:
  for tc in response.tool_calls:
    print(f"search: {tc['args']}")
    result = search_tool.invoke(tc['args'])
    result_text = str(result)[:2000]
    messages.append(ToolMessage(content=result_text, tool_call_id=tc['id']))

  final = llm_with_search.invoke(messages)
  print(f"response: {final.content}")

else:
  print(f"response: {response.content}")

In [43]:
def format_search_results(results, max_summary=100):
  formatted = []
  for r in results:
    content = r.get("content", "")
    summary = content[:max_summary] + "..." if len(content) > max_summary else content
    formatted.append({"url" : r.get('url',''), 'summary' : summary})

  return formatted

results = search_tool.invoke({'query' : 'langchain 툴 호출방법'})
for item in format_search_results(results):
  print(f"{item['url']}")
  print(f"   {item['summary']}")
  print("=="*10)


https://jjnomad.tistory.com/65
   Tool 설정하기

LangGraph에서 Tool을 설정하는 방법은 간단합니다. 먼저, Tool을 정의하고, 그것을 노드에서 사용할 수 있도록 연결해야 합니다.

## 환경설정

...
https://rudaks.tistory.com/entry/langchain-Tool%EB%8F%84%EA%B5%AC-%EC%9D%B4%EB%9E%80
   "4 곱하기 3은 얼마인가?"에서 llm은 곱하기를 하기 위해 `multiply` tool을 사용하기로 결정한다. 이 툴(multiply)을 사용하기 위해 파라미터 2개를 추출해야...
https://wikidocs.net/262582
   ```
# 파이썬 코드를 실행하고 결과를 반환합니다. print(python_tool.invoke("print(100 + 200)")) 
```

```
300
```

아래는 L...


In [45]:
!pip install wikipedia

In [47]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [49]:
wiki_api = WikipediaAPIWrapper(top_k_results=2, doc_content_chars_max=1000, lang='ko')
wiki_tool = WikipediaQueryRun(api_wrapper = wiki_api)

In [50]:
wiki_tool

WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from '/usr/local/lib/python3.12/dist-packages/wikipedia/__init__.py'>, top_k_results=2, lang='ko', load_all_available_meta=False, doc_content_chars_max=1000))

In [52]:
wiki_tool.name

'wikipedia'

In [53]:
wiki_tool.description

'A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.'

In [54]:
wiki_tool.invoke('인공지능')

'Page: 인공지능\nSummary: 인공지능(人工智能, 영어: artificial intelligence, AI)은 인간의 학습능력, 추론능력, 지각능력을 인공적으로 구현하려는 컴퓨터 과학의 세부분야 중 하나이다. 정보공학 분야에 있어 하나의 인프라 기술이기도 하다. 인간을 포함한 동물이 갖고 있는 지능 즉, 자연 지능(natural intelligence)과는 다른 개념이다.\n인간의 지능을 모방한 기능을 갖춘 컴퓨터 시스템이며, 인간의 지능을 기계 등에 인공적으로 시연(구현)한 것이다. 일반적으로 범용 컴퓨터에 적용한다고 가정한다. 이 용어는 또한 그와 같은 지능을 만들 수 있는 방법론이나 실현 가능성 등을 연구하는 과학 기술 분야를 지칭하기도 한다.\n\nPage: 생성형 인공지능\nSummary: 생성형 인공지능(영어: Generative artificial intelligence)은 기존 데이터의 패턴과 구조를 학습하여, 텍스트, 이미지, 오디오, 코드 등 새로운 콘텐츠를 생성할 수 있는 인공지능 기술의 총칭이다. 사용자의 프롬프트라고 불리는 지시문이나 입력을 기반으로 결과물을 도출하는 것이 일반적이다.\n생성형 인공지능은 단순히 데이터를 분류하거나 예측하는 전통적인 분석형 AI와 달리, 데이터 분포를 학습하여 창작을 수행한다. 이는 기계 학습의 한 분야인 딥러닝 기술, 특히 인공신경망에 기반한다.\n2010년대 중반 생성적 적대 신경망의 등장은 사실적인 이미지 생성의 가능성을 열었고, 2017년 트랜스포머 아키텍처의 발표는 자연어 처리 분야의 비약적인 발전을 이끌었다. 2020년대 들어 확산 모델이 고품질 이미지 생성의 주류가 되었으며, OpenAI의 ChatGPT (2022년) 출시를 기점으로 생성형 인공지능 기술은 전 세계적인 주목을 받으며 기술적 특이점 논의를 재점화했다.'

In [57]:
llm_with_wiki = llm.bind_tools([wiki_tool])
query = "모두의 연구소에 대해서 설명해줘"
messages = [HumanMessage(content=query)]
response = llm_with_wiki.invoke(messages)
response

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 91, 'total_tokens': 108, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_998d5473a0', 'id': 'chatcmpl-DXQOBsFqr8f6CzzuWK3xlyZ1aA6tA', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019db507-1977-7800-beba-cc7b6f55c0b9-0', tool_calls=[{'name': 'wikipedia', 'args': {'query': '모두의 연구소'}, 'id': 'call_Uhci1gZ9LQ0EzzNmGb4tVrJe', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 91, 'output_tokens': 17, 'total_tokens': 108, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [64]:
if response.tool_calls:
  for tc in response.tool_calls:
    print(f"wikipedia 검색: {tc['args']}")
    result = wiki_tool.invoke(tc['args'])
    result_text = str(result)[:2000]
    messages.append(ToolMessage(content=result_text, tool_call_id=tc['id']))

    final = llm_with_search.invoke(messages)
    print(f"response: {final.content}")

else:
    print(f"response: {response.content}")

wikipedia 검색: {'query': '모두의 연구소'}


BadRequestError: Error code: 400 - {'error': {'message': "Invalid parameter: messages with role 'tool' must be a response to a preceeding message with 'tool_calls'.", 'type': 'invalid_request_error', 'param': 'messages.[1].role', 'code': None}}

In [92]:
from langchain_core.messages import HumanMessage, ToolMessage, AIMessage

# 🌟 1. 아예 새로운 변수명(chat_history)을 사용해서 과거의 쓰레기값을 회피합니다!
query = "모두의 연구소가 뭐야?"
chat_history = [HumanMessage(content=query)]

# 2. 첫 번째 LLM 호출
response = llm_with_search.invoke(chat_history)

if response.tool_calls:
    # 🌟 3. AI가 "툴 쓸게!" 라고 한 응답을 추가 (이게 chat_history[1]이 됩니다)
    chat_history.append(response)

    for tc in response.tool_calls:
        print(f"wikipedia 검색: {tc['args']}")
        result = wiki_tool.invoke(tc['args'])
        result_text = str(result)[:2000]

        # 🌟 4. 툴 검색 결과를 추가 (이게 chat_history[2]가 됩니다)
        chat_history.append(ToolMessage(content=result_text, tool_call_id=tc['id']))

    # ========================================================
    print("\n--- [디버깅] 깨끗해진 리스트 확인 ---")
    for i, msg in enumerate(chat_history):
        print(f"chat_history[{i}] : {type(msg).__name__}")
    print("----------------------------------\n")
    # 정상 결과 예측:
    # chat_history[0] : HumanMessage
    # chat_history[1] : AIMessage
    # chat_history[2] : ToolMessage
    # ========================================================

    # 5. 최종 답변 생성
    final = llm_with_search.invoke(chat_history)
    print(f"최종 답변: {final.content}")

else:
    print(f"답변: {response.content}")

wikipedia 검색: {'query': '모두의 연구소'}

--- [디버깅] 깨끗해진 리스트 확인 ---
chat_history[0] : HumanMessage
chat_history[1] : AIMessage
chat_history[2] : ToolMessage
----------------------------------

최종 답변: "모두의 연구소"에 대한 현재 정보는 확인되지 않았습니다. 하지만 기본적으로 "모두의 연구소"는 특정한 연구 기관을 의미할 가능성이 큽니다. 비슷한 이름의 기관이나 연구소가 존재할 수 있으니, 더 구체적인 정보를 제공해 주시면 검색하여 더 정확한 내용을 찾아드릴 수 있습니다.


In [91]:
# 오류나는 코드

# from langchain_core.messages import HumanMessage, ToolMessage
# if response.tool_calls:
#     messages.append(response)

#     for tc in response.tool_calls:
#         print(f"wikipedia 검색: {tc['args']}")
#         result = wiki_tool.invoke(tc['args'])
#         result_text = str(result)[:2000]
#         messages.append(ToolMessage(content=result_text, tool_call_id=tc['id']))
#         print("\n--- [디버깅] 에러 방지 체크 ---")
#         for i, msg in enumerate(messages):
#           print(f"messages[{i}] : {type(msg).__name__}")
#         print("------------------------------\n")
#     final = llm_with_search.invoke(messages)
#     print(f"response: {final.content}")




# else:
#     print(f"response: {response.content}")

wikipedia 검색: {'query': '모두의 연구소'}

--- [디버깅] 에러 방지 체크 ---
messages[0] : HumanMessage
messages[1] : ToolMessage
------------------------------



BadRequestError: Error code: 400 - {'error': {'message': "Invalid parameter: messages with role 'tool' must be a response to a preceeding message with 'tool_calls'.", 'type': 'invalid_request_error', 'param': 'messages.[1].role', 'code': None}}

In [71]:
from langchain_core.messages import HumanMessage, ToolMessage
@tool
def extract_key_sentences(text, keyword, max_sentences=3):
  sentences = [s.strip() for s in text.split(".") if s.strip()]
  matched = [s + "." for s in sentences if keyword.lower() in s.lower()]
  return matched[:max_sentences]

ValueError: Function must have a docstring if description not provided.

### 덕덕고

In [74]:
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

In [85]:
ddg_wrapper = DuckDuckGoSearchAPIWrapper(max_results=3)
ddg_tool = DuckDuckGoSearchResults(api_wrapper = ddg_wrapper)

In [78]:
!pip install -U duckduckgo-search

In [81]:
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

# 패키지 설치 후 다시 초기화 시도
try:
    ddg_wrapper = DuckDuckGoSearchAPIWrapper(max_results=3)
    ddg_tool = DuckDuckGoSearchResults(api_wrapper=ddg_wrapper)
    print("DuckDuckGo Tool successfully initialized.")
    display(ddg_tool.invoke("LangChain MCP"))
except Exception as e:
    print(f"Error: {e}")

Error: Could not import ddgs python package. Please install it with `pip install -U ddgs`.


In [86]:
ddg_tool.name, ddg_tool.description

('duckduckgo_results_json',
 'A wrapper around Duck Duck Go Search. Useful for when you need to answer questions about current events. Input should be a search query.')

In [87]:
ddg_tool.invoke("랭체인 튜토리얼 2026")

"snippet: Jan 17, 2026 · 오늘은 파이썬 기초만 있다면 누구나 따라 할 수 있는 **'나만의 지식베이스 AI 챗봇 만들기'**를 2026년 최신 트렌드에 맞춰 **LangChain (랭체인)**으로 구현하는 방법을 A부터 Z까지 깔끔하게 정리해 드리겠습니다., title: LangChain (랭체인)으로 나만의 데이터 학습시키기: 2026년 최신 RAG ..., link: https://panggeria.tistory.com/entry/LangChain랭체인으로-나만의-데이터-학습시키기-2026년-최신-RAG-완벽-가이드, snippet: Nov 22, 2024 · LangChain 튜토리얼 가이드 2 분 소요 LangChain 소개 기본 구성 요소 LLM/Chat Models Groq 사용 OpenAI 사용 Ollama 사용 프롬프트 템플릿 출력 파서 주요 기능 및 사용 예시 체인 구성 비동기 처리 실제 구현 예시 질문 생성 시스템 데이터 검증 시스템 1. LangChain 소개, title: LangChain 튜토리얼 가이드 - gityeop blog, link: https://gityeop.github.io/machine-learning/LangChain-Tutorial.md/, snippet: 이책은 거대언어모델 (LLM)을 활용하여 애플리케이션과 파이프라인을 신속하게 구축할 때 주로 활용되는 랭체인 (LangChain) 프레임워크에 대해 입문부터 응용까지 다룹니다., title: 랭체인 (LangChain) 입문부터 응용까지 [ver 1.0+] - WikiDocs, link: https://wikidocs.net/book/14473, snippet: #랭체인 한국어 튜토리얼🇰🇷 업데이트 소식🔥 처음 사용자를 위한 친절한 환경설치 (Windows, Mac) 테디노트 TeddyNote • 20K views • 1 year ago, title: LangChain 튜토리얼 - YouTube, link: https://www.youtube.c

In [90]:
import time
query = "2026 4월 AI 트렌드"
tools = {"Tavily" : search_tool, "Wikipedia":wiki_tool, "DuckDuckGO": ddg_tool}

for name, tool in tools.items():
  start = time.time()
  try:
    result = tool.invoke({'query': query} if name == 'Tavily' else query)
    elapsed = time.time() - start
    result_str = str(result)
    print(f"{name} : {elapsed:.2f}sec")
    print(f"result length : {len(result_str)}")
    print(f"{result_str[:200]}")

  except:
    print(f"{name}: error")

  print("======")

Tavily : 0.30sec
result length : 7141
[{'title': '2026년 4월 AI 마케팅 트렌드: 매일 아침 10시, 바이럴 콘텐츠 자동화 루틴', 'url': 'https://pro-feeder.tistory.com/entry/2026-ai-viral-content-automation-routine', 'content': '# 마케팅 데이터로 분석하는 자산 관리 전략\n\nAI & 마케팅 테크
Wikipedia : 1.25sec
result length : 1000
Page: 엔비디아
Summary: 엔비디아 코퍼레이션(Nvidia Corporation)은 캘리포니아주 샌타클래라에 본사를 두고 델라웨어에 법인을 둔 미국의 다국적 기업이자 기술 회사이다. 데이터 사이언스 및 고성능 컴퓨팅을 위한 그래픽 처리 장치(GPU), API(애플리케이션 프로그래밍 인터페이스), 모바일 컴퓨팅 및 자동차용 SoC(시스템 온 칩 유닛
DuckDuckGO : 7.11sec
result length : 1185
snippet: 2 days ago - Byprobs 4월 21, 20264월 20, 2026 · 챗봇의 시대는 저물고, 스스로 판단하고 움직이는 ‘에이전트(Agent)’의 시대가 정착되었습니다. 불과 1~2년 전만 해도 신기한 장난감 같았던 AI가 이제는 실질적인 업무 동료가 된 셈이죠., title: 2026년 AI 트렌드: 단순 비서에서 해결사로 진화


In [96]:
import math
# tool 이란 변수를 먼저 써버리면 오류 발생 가능..
# 다시 import
from langchain_core.tools import tool

@tool
def calculator(expression):
  """math expression with eval"""
  try:
    allowed = {'math':math, 'abs': abs, 'round' : round, 'min': min, 'max': max}
    result = eval(expression, {"__builtins__": {}}, allowed)
    return str(result)

  except:
    return f"calculation error"

In [97]:
calculator.invoke({'expression': 'math.sqrt(144) + math.pi'})

'15.141592653589793'

In [98]:
!pip install langchain_experimental

In [99]:
# REPL
from langchain_experimental.tools import PythonREPLTool

In [100]:
python_tool = PythonREPLTool()

In [101]:
python_tool.name, python_tool.description

('Python_REPL',
 'A Python shell. Use this to execute python commands. Input should be a valid python command. If you want to see the output of a value, you should print it out with `print(...)`.')

In [119]:
python_tool.invoke("print(sum(range(1,101)))")

'5050\n'

In [103]:
code_str = """
import math
data = [23, 45, 12, 67, 34, 89, 56]
avg = sum(data) / len(data)
std = math.sqrt(sum((x-avg)**2 for x in data) / len(data))
print(f"평균 : {avg:.1f}, 표준편차 : {std:.1f}")

"""

In [104]:
code_str

'\nimport math\ndata = [23, 45, 12, 67, 34, 89, 56]\navg = sum(data) / len(data)\nstd = math.sqrt(sum((x-avg)**2 for x in data) / len(data))\nprint(f"평균 : {avg:.1f}, 표준편차 : {std:.1f}")\n\n'

In [105]:
python_tool.invoke(code_str)

'평균 : 46.6, 표준편차 : 24.5\n'

In [107]:
llm_with_python = llm.bind_tools([python_tool])
query = "피보나치 수열의 처음 10개 항을 계산하고 출력해줘"
messages = [HumanMessage(content=query)]
response = llm_with_python.invoke(messages)
messages.append(response)

In [108]:
response

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 96, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a7190374f3', 'id': 'chatcmpl-DXRAvoXlLLAn4qqrS1ROsbzSEKVF0', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019db535-3374-7ad1-9600-54e4e7f84306-0', tool_calls=[{'name': 'Python_REPL', 'args': {'query': 'def fibonacci(n):\n    fib_sequence = []\n    a, b = 0, 1\n    for _ in range(n):\n        fib_sequence.append(a)\n        a, b = b, a + b\n    return fib_sequence\n\nfibonacci(10)'}, 'id': 'call_DxGCrdjYa65opTkMdsQzh5sC', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_t

In [109]:
messages

[HumanMessage(content='피보나치 수열의 처음 10개 항을 계산하고 출력해줘', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 96, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_a7190374f3', 'id': 'chatcmpl-DXRAvoXlLLAn4qqrS1ROsbzSEKVF0', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019db535-3374-7ad1-9600-54e4e7f84306-0', tool_calls=[{'name': 'Python_REPL', 'args': {'query': 'def fibonacci(n):\n    fib_sequence = []\n    a, b = 0, 1\n    for _ in range(n):\n        fib_sequence.append(a)\n        a, b = b, a + b\n    return fib_sequence\n\nfibonacci(10)'}, 'id': 'ca

In [116]:
if response.tool_calls:
  for tc in response.tool_calls:
    print("code execution")
    print(tc['args'].get('query', "")) # 코드에 있는 query가 아래 tc['args']의 query에 들어갈 것
    result = python_tool.invoke(tc['args'])
    print(f"output : {result}")
    messages.append(ToolMessage(content=str(result), tool_call_id = tc['id']))

  final = llm_with_python.invoke(messages)

final.content

code execution
def fibonacci(n):
    fib_sequence = []
    a, b = 0, 1
    for _ in range(n):
        fib_sequence.append(a)
        a, b = b, a + b
    return fib_sequence

fibonacci(10)
output : 


BadRequestError: Error code: 400 - {'error': {'message': "Invalid parameter: messages with role 'tool' must be a response to a preceeding message with 'tool_calls'.", 'type': 'invalid_request_error', 'param': 'messages.[3].role', 'code': None}}

In [113]:
print(tc['args'])

{'query': 'def fibonacci(n):\n    fib_sequence = []\n    a, b = 0, 1\n    for _ in range(n):\n        fib_sequence.append(a)\n        a, b = b, a + b\n    return fib_sequence\n\nfibonacci(10)'}


In [ ]:
if response.tool_calls:
  for tc in response.tool_calls:
    print("code execution")
    print(tc['args'].get('query', "")) # 코드에 있는 query가 아래 tc['args']의 query에 들어갈 것
    result = python_tool.invoke(tc['args'])
    print(f"output : {result}")
    messages.append(ToolMessage(content=str(result), tool_call_id = tc['id']))

  final = llm_with_python.invoke(messages)


In [117]:
python_tool = PythonREPLTool()
llm_with_python = llm.bind_tools([python_tool])


def run_code_agent(question, llm_with_tools, python_tool):
  llm_with_tools = llm.bind_tools([python_tool])
  llm_with_tools.invoke(question)


  return {"code": "", "output": "", "answer" : response.content}

result = run_code_agent("1부터 100까지의 소수를 구해줘", llm_with_python, python_tool)

In [118]:
result

{'code': '', 'output': '', 'answer': ''}

In [128]:
def run_code_agent(question, llm_with_tools, python_tool):
  messages = [HumanMessage(content=question)]
  response = llm_with_tools.invoke(messages)
  messages.append(response)

  code_text = ""
  output = ""

  if response.tool_calls:
    for tc in response.tool_calls:
      code_text = tc['args'].get('query', '')
      output = str(python_tool.invoke(tc['args']))
      messages.append(ToolMessage(content=output, tool_call_id = tc['id']))

    final = llm_with_tools.invoke(messages)
    return {"code": code_text, "output": output.strip(), "answer": final.content}

  return {"code": code_text, "output": output.strip(), "answer": final.content}

In [123]:

def run_code_agent(question, llm, python_tool):
    # 1. LLM에 파이썬 툴 연결
    llm_with_tools = llm.bind_tools([python_tool])

    # 2. 대화 기록 초기화 (이전 에러 해결에서 배운 방식!)
    messages = [HumanMessage(content=question)]

    # 3. LLM 첫 번째 호출 (코드 작성 요청)
    response = llm_with_tools.invoke(question)

    # 반환할 값들을 미리 빈 문자열로 세팅
    generated_code = ""
    execution_output = ""
    final_answer = ""

    # 4. LLM이 파이썬 코드를 작성하고 툴을 호출한 경우
    if response.tool_calls:
        # 필수: LLM의 툴 호출 응답을 메시지에 추가
        messages.append(response)

        for tc in response.tool_calls:
            # 툴에 전달된 인자(args)에서 작성된 파이썬 코드를 추출
            # (보통 LangChain 파이썬 툴은 'query' 또는 'code'라는 키에 코드를 담습니다)
            args_dict = tc['args']
            # 딕셔너리 값들 중 코드를 찾아 문자열로 저장
            generated_code += str(args_dict.get('query', args_dict.get('code', args_dict)))

            # 파이썬 코드 실행 (툴 invoke)
            result = python_tool.invoke(args_dict)
            execution_output += str(result)

            # 실행 결과를 메시지에 추가
            messages.append(ToolMessage(content=str(result), tool_call_id=tc['id']))

        # 5. 실행 결과를 바탕으로 최종 답변 생성 (두 번째 호출)
        final = llm_with_tools.invoke(messages)
        final_answer = final.content

    # 6. LLM이 툴을 안 쓰고 바로 답변한 경우 (일반 대화)
    else:
        final_answer = response.content

    # 7. 요구하신 딕셔너리 형태로 반환
    return {
        "code": generated_code,       # LLM이 작성한 파이썬 코드
        "output": execution_output,   # 코드를 실행한 결과값 (에러 메시지 혹은 print 출력 등)
        "answer": final_answer        # 실행 결과를 보고 LLM이 해석해준 최종 답변
    }

In [124]:
python_tool = PythonREPLTool()
llm_with_python = llm.bind_tools([python_tool])

result = run_code_agent("1부터 100까지의 소수를 구해줘", llm_with_python, python_tool)
result

{'code': 'def is_prime(num):\n    if num < 2:\n        return False\n    for i in range(2, int(num**0.5) + 1):\n        if num % i == 0:\n            return False\n    return True\n\nprimes = [num for num in range(1, 101) if is_prime(num)]\nprint(primes)',
 'output': '[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97]\n',
 'answer': '1부터 100까지의 소수는 다음과 같습니다:\n\n\\[ 2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97 \\]'}

In [129]:
result = run_code_agent("1부터 100까지의 소수를 구해줘", llm_with_python, python_tool)
result

{'code': 'def is_prime(n):\n    if n <= 1:\n        return False\n    for i in range(2, int(n**0.5) + 1):\n        if n % i == 0:\n            return False\n    return True\n\nprimes = [n for n in range(1, 101) if is_prime(n)]\nprint(primes)',
 'output': '[2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97]',
 'answer': '1부터 100까지의 소수는 다음과 같습니다:\n\n\\[ 2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97 \\]'}

In [130]:
print(f"code: {result['code']}")
print(f"output: {result['output']}")
print(f"answer: {result['answer']}")

code: def is_prime(n):
    if n <= 1:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

primes = [n for n in range(1, 101) if is_prime(n)]
print(primes)
output: [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97]
answer: 1부터 100까지의 소수는 다음과 같습니다:

\[ 2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71, 73, 79, 83, 89, 97 \]


In [132]:
all_tools = [search_tool, wiki_tool, ddg_tool, calculator]
llm_agent = llm.bind_tools(all_tools)

In [140]:
# c-rag
# self-rag
# graph-rag

# mcp, agent 등..

tool_map = {t.name:t for t in all_tools}
for t in all_tools:
  print(f" tools : {t.name}: {t.description}")

def agent_loop(query, llm_agent, tool_map, max_turns = 3):
  messages = [HumanMessage(content = query)]
  tools_used = []

  for turn in range(max_turns):
    response = llm_agent.invoke(messages)
    messages.append(response)

    if not response.tool_calls:

      return {'query': query, 'tools_used': tools_used}
      #return response.content

    for tc in response.tool_calls:
      tool_name = tc['name']
      print(f" [{turn+1}] {tool_name}({tc['args']})")
      tool_fn = tool_map.get(tool_name)
      if tool_fn:
        result = tool_fn.invoke(tc['args'])

      else:
        result = f"도구 {tool_name}를 찾을 수 없습니다."

      messages.append(ToolMessage(content=str(result)[:2000], tool_call_id = tc['id']))

  final = llm_agent.invoke(messages)
  return final.content

 tools : tavily_search_results_json: A search engine optimized for comprehensive, accurate, and trusted results. Useful for when you need to answer questions about current events. Input should be a search query.
 tools : wikipedia: A wrapper around Wikipedia. Useful for when you need to answer general questions about people, places, companies, facts, historical events, or other subjects. Input should be a search query.
 tools : duckduckgo_results_json: A wrapper around Duck Duck Go Search. Useful for when you need to answer questions about current events. Input should be a search query.
 tools : calculator: math expression with eval


In [141]:
test_queries = [
    '2026년 AI 스타트업 투자 동향을 알려줘',
    '랭체인이 뭔지 설명해줘',
    '원주율 x 100의 제곱근은?'
]

for q in test_queries:
  answer = agent_loop(query, llm_agent, tool_map, max_turns = 3)
  print(answer)
  print("="*10)

 [1] calculator({'expression': '[0, 1].concat(Array.from({length: 8}, (_, i) => { return i < 2 ? i : [0, 1][i-2] + [0, 1][i-1]; }))'})
 [2] calculator({'expression': 'let fib = [0, 1]; for (let i = 2; i < 10; i++) { fib[i] = fib[i - 1] + fib[i - 2]; } fib'})
 [3] calculator({'expression': 'let fib = [0, 1]; for (let i = 2; i < 10; i++) { fib[i] = fib[i - 1] + fib[i - 2]; } fib'})

 [1] calculator({'expression': '[0, 1].concat(Array.from({length: 8}, (_, i) => (i < 2 ? i : [0, 1].slice(-2).reduce((a, b) => a + b))) )'})
 [2] calculator({'expression': '(function(n){let arr=[0,1];for(let i=2;i<n;i++){arr.push(arr[i-1]+arr[i-2])}return arr})(10)'})
 [3] calculator({'expression': '(function(n){let arr=[0, 1];for(let i=2;i<n;i++){arr[i] = arr[i-1] + arr[i-2];}return arr})(10)'})
피보나치 수열의 처음 10개 항을 수동으로 계산하여 보겠습니다.

피보나치 수열은 각 항이 이전 두 항의 합인 수열입니다. 시작값은 0과 1입니다.

1. F(0) = 0
2. F(1) = 1
3. F(2) = F(0) + F(1) = 0 + 1 = 1
4. F(3) = F(1) + F(2) = 1 + 1 = 2
5. F(4) = F(2) + F(3) = 1 + 2 = 3
6. F(5

In [142]:
def search_and_summarize(query, search_tool, llm, max_results=3):
  # search
  raw_results = search_tool.invoke({'query': query})

  # source
  sources = []
  combined_text = ""
  for r in raw_results[:max_results]:
    url = r.get('url', 'no source')
    content = r.get('content', '')
    sources.append(url)
    combined_text += f"[출처: {url}] \n {content}\n\n"


  summary_prompt = f"""다음 검색 결과를 바탕으로 '{query}'에 대한 리포트를 작성하세요.

  검색 결과:
  {combined_text[:3000]}

  리포트 형식:
  1. 핵심 요약 (2~3문장)
  2. 주요 포인트 (3개)
  3. 출처 목록"""

  response = llm.invoke([HumanMessage(content = summary_prompt)])

  return {
      'query' : query,
      'sources' : sources,
      'report' : response.content,
      'raw_result_count' : len(raw_results)

  }

In [143]:
search_and_summarize("2026년 생성형 AI 시장 전망", search_tool, llm)

{'query': '2026년 생성형 AI 시장 전망',
 'sources': ['https://www.seroai.xyz/posts/2026-04-01-ai-2026',
  'https://aimatters.co.kr/news-report/ai-report/2705',
  'https://www.goldspoonlab.com/uncategorized/2026-generative-ai-market-growth'],
 'report': '# 2026년 생성형 AI 시장 전망 리포트\n\n## 핵심 요약\n2026년 글로벌 생성형 AI 시장은 전년 대비 340% 성장하여 1,300억 달러 규모에 이를 것으로 예상된다. 기업용 솔루션과 개인용 AI 도구의 대중화가 동시에 진행되며, AI 기술의 실용화가 가속화되고 있다. 규제 환경 변화와 함께 지속적인 기술 발전이 시장의 주요 이슈로 부각될 전망이다.\n\n## 주요 포인트\n1. **폭발적 성장**: 2026년 생성형 AI 시장 규모는 1,300억 달러에 달하며, 이는 예상보다 3배 이상 빠른 성장률을 나타낸다. 이로 인해 AI에 대한 투자와 관심이 급증하고 있다.\n   \n2. **투자 동향**: 2026년 1분기 AI 스타트업에 대한 투자액이 580억 달러에 이르며, 이 중 AI 인프라, 생성형 AI 앱, AI 반도체 영역에서 각각 큰 비중을 차지하고 있다. 기업 공개상장(IPO)의 평균 상승률도 178%로, 투자자들의 높은 관심을 반영한다.\n\n3. **규제와 변화**: AI 규제 정책의 구체화가 2026년 하반기의 주요 이슈로 부각될 예정이다. 특히 유럽의 AI Act 시행은 글로벌 AI 기업들의 운영 비용에 큰 영향을 미칠 것으로 보인다. 데이터 프라이버시 우려와 개발도상국에서의 인식 부족도 시장 성장을 제한하는 요소로 작용할 수 있다.\n\n## 출처 목록\n1. Sero AI, "2026년 생성형 AI 시장 전망", 2026년 4월 1일, [Sero AI](https://www.seroai.xyz/